# Quick Maths with Matrices!
---

<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Quick-Maths-with-Matrices!" data-toc-modified-id="Quick-Maths-with-Matrices!-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Quick Maths with Matrices!</a></span><ul class="toc-item"><li><span><a href="#Import-Libraries" data-toc-modified-id="Import-Libraries-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Import Libraries</a></span></li><li><span><a href="#Test-Framework" data-toc-modified-id="Test-Framework-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Test Framework</a></span></li></ul></li><li><span><a href="#Optimizing-Matrix-Multiplications" data-toc-modified-id="Optimizing-Matrix-Multiplications-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Optimizing Matrix Multiplications</a></span><ul class="toc-item"><li><span><a href="#For-Loop" data-toc-modified-id="For-Loop-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>For Loop</a></span></li><li><span><a href="#Array-Slicing" data-toc-modified-id="Array-Slicing-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Array Slicing</a></span></li><li><span><a href="#Improvement-with-array-slicing" data-toc-modified-id="Improvement-with-array-slicing-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Improvement with array slicing</a></span></li><li><span><a href="#Array-Broadcasting" data-toc-modified-id="Array-Broadcasting-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Array Broadcasting</a></span></li><li><span><a href="#Improvement-with-array-broadcasting" data-toc-modified-id="Improvement-with-array-broadcasting-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Improvement with array broadcasting</a></span></li><li><span><a href="#Einstein-Sum" data-toc-modified-id="Einstein-Sum-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Einstein Sum</a></span></li><li><span><a href="#Improvement-with-einstein-sum" data-toc-modified-id="Improvement-with-einstein-sum-2.7"><span class="toc-item-num">2.7&nbsp;&nbsp;</span>Improvement with einstein sum</a></span></li><li><span><a href="#Linear-Algebra-Libraries" data-toc-modified-id="Linear-Algebra-Libraries-2.8"><span class="toc-item-num">2.8&nbsp;&nbsp;</span>Linear Algebra Libraries</a></span></li><li><span><a href="#Improvement-with-linear-algebra-libraries" data-toc-modified-id="Improvement-with-linear-algebra-libraries-2.9"><span class="toc-item-num">2.9&nbsp;&nbsp;</span>Improvement with linear algebra libraries</a></span></li></ul></li></ul></div>

## Import Libraries

In [1]:
import torch
import timeit
import operator
from functools import partial

## Test Framework

In [2]:
def test(a, b, compare, compare_name=None):
    if compare_name is None:
        compare_name = compare.__name__
    assert compare(a, b),\
    f"{compare_name} check failed:\n{a}\n{b}"

def test_equality(a, b):
    test(a, b, operator.eq, "Equality")

def test_approximately(a, b):
    allclose = partial(torch.allclose, atol=1e-5, rtol=1e-03)
    if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
        a = torch.tensor(a)
        b = torch.tensor(b)
    test(a, b, allclose, "Approximate Equality")

In [3]:
test_equality(1e-5,1e-5)

In [4]:
test_approximately(1e-5, 1e-6)

# Optimizing Matrix Multiplications

**Test Variables**

In [5]:
A = torch.randn([10,10])
B = torch.randn([10,10])

In [6]:
(A@B).shape

torch.Size([10, 10])

## For Loop

In [7]:
def matmul(A,B):
    A_rows, A_cols = A.shape
    B_rows, B_cols = B.shape
    assert A_cols==B_rows,\
    f"Inner dimensions must match: {A_cols} not equal to {B_rows}"
    C = torch.zeros([A_rows, B_cols])
    for i in range(A_rows):
        for j in range(B_cols):
            for k in range(A_cols):
                C[i,j] += A[i,k] * B[k,j]
    return C

In [8]:
matmul(A,B)

tensor([[ 2.0418, -2.7536, -4.4085, -1.6034, -0.3382, -3.1131, -6.8044,  2.4283,
         -2.9141,  2.4832],
        [ 1.0657,  1.8025,  1.5444, -0.1167,  4.6297,  0.4264, -1.1194, -1.2556,
          4.7425,  4.9186],
        [-2.0994, -2.0461, -0.2605, -3.0962,  1.5434, -0.6749, -4.9513,  0.5862,
         -0.9482,  1.8264],
        [ 3.5155, -1.0560,  2.0752,  3.5571, -0.3114,  2.6617, -0.0998,  4.5250,
         -0.3525, -2.2175],
        [ 2.5120,  1.5429, -0.5752,  0.9798, -2.4916,  0.5847, -0.5694, -0.1662,
          0.8110, -3.2612],
        [-0.2043, -2.5537,  3.4442, -7.4387, -1.1635, -0.6491, -4.4388, -0.1008,
          1.7494, -5.8961],
        [ 3.9631,  5.8521, -3.1028, -1.8217,  3.2243,  2.1090, -0.3819, -1.3312,
          0.8423,  3.0483],
        [-0.6465,  5.9780,  0.8486, -4.1447,  2.0708,  0.3412,  4.7089, -2.0006,
          3.7259, -1.5503],
        [ 2.3947, -3.6057,  1.8346, -0.7736, -1.6121, -1.5473,  1.6892,  4.5211,
         -2.3727, -1.0807],
        [ 1.3023,  

In [9]:
test_approximately(matmul(A, B), (A@B))

In [10]:
matmul_loop_time = timeit.timeit(partial(matmul,A,B), number=10)
matmul_loop_time

0.19198706399998855

In [11]:
# Call the matrix multiplication function
C = matmul(A, B)
print("Result Matrix C (A x B):")
print(C)

Result Matrix C (A x B):
tensor([[ 2.0418, -2.7536, -4.4085, -1.6034, -0.3382, -3.1131, -6.8044,  2.4283,
         -2.9141,  2.4832],
        [ 1.0657,  1.8025,  1.5444, -0.1167,  4.6297,  0.4264, -1.1194, -1.2556,
          4.7425,  4.9186],
        [-2.0994, -2.0461, -0.2605, -3.0962,  1.5434, -0.6749, -4.9513,  0.5862,
         -0.9482,  1.8264],
        [ 3.5155, -1.0560,  2.0752,  3.5571, -0.3114,  2.6617, -0.0998,  4.5250,
         -0.3525, -2.2175],
        [ 2.5120,  1.5429, -0.5752,  0.9798, -2.4916,  0.5847, -0.5694, -0.1662,
          0.8110, -3.2612],
        [-0.2043, -2.5537,  3.4442, -7.4387, -1.1635, -0.6491, -4.4388, -0.1008,
          1.7494, -5.8961],
        [ 3.9631,  5.8521, -3.1028, -1.8217,  3.2243,  2.1090, -0.3819, -1.3312,
          0.8423,  3.0483],
        [-0.6465,  5.9780,  0.8486, -4.1447,  2.0708,  0.3412,  4.7089, -2.0006,
          3.7259, -1.5503],
        [ 2.3947, -3.6057,  1.8346, -0.7736, -1.6121, -1.5473,  1.6892,  4.5211,
         -2.3727, -1.0

In [12]:
matmul_time = timeit.timeit(partial(matmul, A, B), number=1)
print(f"Time taken for matmul(A, B)T4 GPU{matmul_time:.6f} seconds")

Time taken for matmul(A, B)T4 GPU0.020281 seconds
